In [1]:
# Import some libraries and especially load environment variables
from langchain_openai  import AzureChatOpenAI
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
# set an environment varibale
os.environ["LANGCHAIN_PROJECT"] = "01_basic"

print(os.getenv("AZURE_ENDPOINT"))
print(os.getenv("OPENAI_API_BASE"))

https://alkopenai2.openai.azure.com/
https://alkopenai2.openai.azure.com/


In [2]:
llm = AzureChatOpenAI(
	temperature=0,
    openai_api_version="2023-09-01-preview",
    deployment_name="GPT-4o-mini", #Deployment name
    azure_endpoint=os.environ["OPENAI_API_BASE"],
    model_name="GPT-4o-mini"
)

In [3]:
# I the import my custom function to work with audio and videp
from plugins.CallJarvis import Tasks
tasks = Tasks()


In [4]:

# you can test the function
result = tasks.change_task_title("Task_1", "new title")
# print(f"New version of the task: {result}")

Changing title of task Task_1 to new title


In [5]:
# Then create tools list
from langchain.tools import  StructuredTool

tools = [
    StructuredTool.from_function(
        func=tasks.change_task_title,
        name="ChangeTaskTitle",
        description="Change the title of a task.",
        args_schema=tasks.ChangeTaskTitleInput
    ),
    StructuredTool.from_function(
        func=tasks.search_task,
        name="SearchTask",
        description="Can search task with a full Text string.",
        args_schema=tasks.SearchTaskInput
    ),
    StructuredTool.from_function(
        func=tasks.load_task,
        name="LoadTask",
        description="Given a task_id can load task detail in json format.",
        args_schema=tasks.LoadTaskInput
    ),
    StructuredTool.from_function(
        func=tasks.load_tasks,
        name="LoadTasks",
        description="Can load multiple tasks given a list of id in json format. PRefer this if you need to load multiple tasks",
        args_schema=tasks.LoadTasksInput
    )
]

In [6]:
from langchain.agents import create_tool_calling_agent
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
agent = create_tool_calling_agent(tools=tools, llm=llm, prompt=prompt)

/Users/gianmariaricci/develop/github/ai-playground/src/python/various/various/lib/python3.12/site-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [7]:
# Everthing is ready to start the conversation
from langchain.agents import AgentExecutor
from pprint import pprint   

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result =agent_executor.invoke({"input": """
Search all task containing "verdolazzy" and if the due date is in year 2021 add the word "BOMB" as prefix of the title
"""})

pprint(result)



> Entering new AgentExecutor chain...

Invoking: `SearchTask` with `{'search_string': 'verdolazzy'}`


Searching for task verdolazzy
['Task_3', 'Task_4', 'Task_6']
Invoking: `LoadTasks` with `{'task_ids': ['Task_3', 'Task_4', 'Task_6']}`


Loading tasks ['Task_3', 'Task_4', 'Task_6']
get tasks returned [{"id": "Task_3", "title": "Hey I'm task 3", "description": "This is the description of task 3", "version": 5, "due_date": "2021-10-10"}, {"id": "Task_4", "title": "I'm beautiful task 4", "description": "This is the description of task 4", "version": 5, "due_date": "2022-10-10"}, {"id": "Task_6", "title": "I'm the six", "description": "This is the description of task 6", "version": 5, "due_date": "2023-10-10"}]
[{"id": "Task_3", "title": "Hey I'm task 3", "description": "This is the description of task 3", "version": 5, "due_date": "2021-10-10"}, {"id": "Task_4", "title": "I'm beautiful task 4", "description": "This is the description of task 4", "version": 5, "due_date": "2022-10-10"}